## Quiet the logs

Disable DEBUG logging to avoid noisy output from redis-py, httpx, etc.

In [1]:
import logging

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger().setLevel(logging.WARNING)
for _noisy in ("redis", "httpx", "httpx2", "httpcore", "httpcore2",
               "urllib3", "asyncio", "mcp"):
    logging.getLogger(_noisy).setLevel(logging.WARNING)

print("log level: WARNING (set INFO or DEBUG here if you want the detail back)")

log level: WARNING (set INFO or DEBUG here if you want the detail back)


## Chi-squared test for a variance, across the fleet

`chi2_test` asks whether the fleet's variance of a column equals a
hypothesised `sigma0_sq`:

    chi2 = (n - 1) * s^2 / sigma0^2        with n - 1 degrees of freedom

Two things make this worth a dedicated skill rather than hand-composition.

The `variance` step returns the **population** variance (ddof=0), and the
test is defined on the **sample** variance. Since `s^2 = var_pop*n/(n-1)`,
the `(n-1)` cancels and the statistic is just `n*var_pop/sigma0^2` - which
is why the plan looks shorter than the formula. Composed by hand without
that conversion, the head produced 34.4160, 35.4046, and once a p-value
estimated from memory, where the answer is 35.7659. All three reached the
right *conclusion*, which is why the wrong statistics went unnoticed.

This is **not** a goodness-of-fit test. An earlier skill held this name
and computed `sum((x-mu)^2/mu)`, which is a chi-squared statistic only for
Poisson counts; it was removed.

## Parameters

In [2]:
import os, time, json, subprocess

HEAD_APP_ID   = os.environ.get("HEAD_APP_ID", "scarlet-agent-head")
WORKER_APP_ID = os.environ.get("WORKER_APP_ID", "scarlet-agent-worker")

HEAD_NODE    = os.environ.get("HEAD_NODE_ADDRESS", "head")
WORKER_NODES = [n.strip() for n in
                os.environ.get("WORKER_NODE_ADDRESSES", "worker1,worker2,worker3,worker4").split(",")
                if n.strip()]

HEAD_BUS      = os.environ.get("HEAD_BUS", f"{HEAD_APP_ID}_headagent")
HEAD_HOST     = os.environ.get("HEAD_HOST", "agent-head")
HEAD_MCP_PORT = os.environ.get("HEAD_MCP_PORT", "8090")

_explicit_ids = os.environ.get("EXPECTED_AGENT_IDS", "").strip()
if _explicit_ids:
    EXPECTED_AGENT_IDS = {i.strip() for i in _explicit_ids.split(",") if i.strip()}
    HEAD_AGENT_ID      = f"{HEAD_APP_ID}_{HEAD_NODE}"
    WORKER_AGENT_IDS   = EXPECTED_AGENT_IDS - {HEAD_AGENT_ID}
else:
    HEAD_AGENT_ID      = f"{HEAD_APP_ID}_{HEAD_NODE}"
    WORKER_AGENT_IDS   = {f"{WORKER_APP_ID}_{n}" for n in WORKER_NODES}
    EXPECTED_AGENT_IDS = {HEAD_AGENT_ID} | WORKER_AGENT_IDS

GUSTAVO_MANAGED_AGENTS = os.environ.get("GUSTAVO_MANAGED_AGENTS", "false").lower() == "true"
COMPOSE_FILE = os.environ.get("COMPOSE_FILE_PATH", "/workspace/docker-compose.yml")

DATA_DIR = os.environ.get("DATA_DIR", "/data")

print(f"head app id   : {HEAD_APP_ID}")
print(f"worker app id : {WORKER_APP_ID}")
print(f"workers       : {WORKER_NODES}")
print(f"fleet managed : {'Gustavo' if GUSTAVO_MANAGED_AGENTS else 'this notebook (docker compose)'}")

head app id   : loop-agent-head
worker app id : loop-agent-worker
workers       : ['worker1', 'worker2', 'worker3', 'worker4']
fleet managed : this notebook (docker compose)


## Generate each worker's matrix data and its local config

Four workers, four different shapes, four different column sets. The
intersection is never written down anywhere — it has to be discovered.

In [3]:
import numpy as np
import pandas as pd
import yaml

LAYOUT = {
    "worker1": ("plant_alpha/turbine_vibration.csv",  "turbine_alpha_vibration",  80,
                ["vibration_rms", "bearing_temp_c", "shaft_speed_rpm", "torque_nm", "power_kw"]),
    "worker2": ("feeds/grid_telemetry.csv",           "grid_telemetry_feed",     100,
                ["vibration_rms", "bearing_temp_c", "shaft_speed_rpm", "power_kw"]),
    "worker3": ("archive/2026/q3/unit7_sensors.csv",  "unit7_archive_q3",        120,
                ["vibration_rms", "bearing_temp_c", "shaft_speed_rpm", "torque_nm", "power_kw",
                 "ambient_temp_c"]),
    "worker4": ("raw_drop/sensor_export_FINAL.csv",   "bulk_sensor_drop",         90,
                ["vibration_rms", "shaft_speed_rpm", "power_kw"]),
}

PARAMS = {
    "vibration_rms":   (2.5,    0.6),
    "bearing_temp_c":  (65.0,   8.0),
    "shaft_speed_rpm": (1800.0, 120.0),
    "torque_nm":       (340.0,  45.0),
    "power_kw":        (64.0,   11.0),
    "ambient_temp_c":  (22.0,   4.0),
}

for i, (node, (relpath, source_name, rows, cols)) in enumerate(LAYOUT.items(), start=1):
    rng = np.random.default_rng(seed=i * 1000)
    start = pd.Timestamp("2026-01-01T00:00:00")
    frame = {"timestamp": [start + pd.Timedelta(minutes=k) for k in range(rows)]}
    for c in cols:
        mean, sd = PARAMS[c]
        frame[c] = np.clip(rng.normal(mean, sd, size=rows), 0.0, None)
    df = pd.DataFrame(frame)

    gap_cols = list(rng.permutation(cols))[:3]
    gap_rows = rng.choice(rows, size=3, replace=False)
    for r, c in zip(gap_rows, gap_cols):
        df.at[r, c] = np.nan

    path = os.path.join(DATA_DIR, relpath)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, index=False)

    scarlet_dir = os.path.join(DATA_DIR, node, ".scarlet")
    os.makedirs(scarlet_dir, exist_ok=True)
    actual_gaps = sorted(df.columns[df.isna().any()].tolist())
    with open(os.path.join(scarlet_dir, "config.yaml"), "w") as fh:
        yaml.safe_dump({"sources": [{
            "name": source_name, "type": "csv", "mode": "local",
            "description": (f"{rows} rows. Columns: {', '.join(cols)}. "
                            f"gaps in columns: {', '.join(actual_gaps)}"),
            "path": path,
        }]}, fh, sort_keys=False)

    has = "has torque_nm" if "torque_nm" in cols else "NO torque_nm"
    print(f"{source_name:26s} {rows:>4} rows  {has}")

SHARED = ["vibration_rms", "shaft_speed_rpm", "power_kw"]
MU0 = 60.0

turbine_alpha_vibration      80 rows  has torque_nm
grid_telemetry_feed         100 rows  NO torque_nm
unit7_archive_q3            120 rows  has torque_nm
bulk_sensor_drop             90 rows  NO torque_nm


## Ground truth, from the same files the workers read

Computed here with pandas over the same CSVs, so the fleet's answer can be
checked rather than believed. `sigma0_sq` is deliberately set well above
the real variance, so the test should reject decisively on the lower
tail - a direction a two-sided-only implementation would get wrong by a
factor of two.

In [4]:
import pandas as pd

SHARED = ["vibration_rms", "shaft_speed_rpm", "power_kw"]
SIGMA0_SQ = 1.0

pooled = pd.concat(
    [pd.read_csv(f"{DATA_DIR}/{rel}")[SHARED].dropna(how="any")
     for rel, _, _, _ in LAYOUT.values()],
    ignore_index=True)
n = len(pooled)
print(f"n = {n}   sigma0_sq = {SIGMA0_SQ}   df = {n - 1}")
for c in SHARED:
    s2 = pooled[c].var(ddof=1)
    chi2 = (n - 1) * s2 / SIGMA0_SQ
    print(f"  {c:<18} s^2 = {s2:>14.4f}   expected chi2 = {chi2:>16.4f}")


n = 380   sigma0_sq = 1.0   df = 379
  vibration_rms      s^2 =         0.3585   expected chi2 =         135.8601
  shaft_speed_rpm    s^2 =     14509.4356   expected chi2 =     5499076.0906
  power_kw           s^2 =       114.2631   expected chi2 =       43305.7066


## Ask the fleet

No source, no column, no SQL.

In [5]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


async def ask_scarlet_agent(message: str) -> str:
    url = f"http://{HEAD_HOST}:{HEAD_MCP_PORT}/mcp"
    async with streamable_http_client(url) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            result = await session.call_tool("ask_scarlet_agent", {"message": message})
            return result.content[0].text


answer = await ask_scarlet_agent(
    f"Run a chi-squared test for a variance across the fleet using the "
    f"chi2_test skill with sigma0_sq={SIGMA0_SQ}. Report the chi2 "
    f"statistic, the degrees of freedom and the p-value for each column, "
    f"and state n once. Do not name any data source or column yourself.")
print(answer)


The chi-squared test for variance was run across the fleet with hypothesised variance σ₀² = 1.0.

- **Degrees of freedom (df)**: 379  
- **Sample size (n)**: 380  

Per-column results:

| Column | χ² statistic | p-value (two-sided) |
|--------|--------------|---------------------|
| power_kw | 43,305.71 | 0.0 |
| shaft_speed_rpm | 5,499,076.09 | 0.0 |
| vibration_rms | 135.86 | 0.0 |

All p-values are effectively 0 (except for vibration_rms, where the upper-tail p-value is 1.0, indicating the observed variance is *less* than σ₀², but still highly significant in a two-sided test). This means the fleet's variances for all three columns differ significantly from 1.0.


## Check it

`df` is the one to read first: it must be `n - 1`. A `df` of `n` means the
plan spent no degree of freedom on the estimated variance, and a `df` far
below `n` means the fleet aggregated fewer rows than it should have.

The statistic below is printed next to the one a plan that forgot the
ddof conversion would produce. They differ by about 1%, which is exactly
the kind of gap that survives review.

In [6]:
print(f"n = {n}   df should be {n - 1}")
for c in SHARED:
    s2_sample = pooled[c].var(ddof=1)
    s2_pop    = pooled[c].var(ddof=0)
    correct = (n - 1) * s2_sample / SIGMA0_SQ
    wrong   = (n - 1) * s2_pop    / SIGMA0_SQ      # forgot n/(n-1)
    print(f"  {c:<18} correct chi2={correct:>16.4f}   "
          f"wrong (ddof=0)={wrong:>16.4f}")


n = 380   df should be 379
  vibration_rms      correct chi2=        135.8601   wrong (ddof=0)=        135.5026
  shaft_speed_rpm    correct chi2=    5499076.0906   wrong (ddof=0)=    5484604.8378
  power_kw           correct chi2=      43305.7066   wrong (ddof=0)=      43191.7443


## All three tails

`mode` selects which tail the alternative hypothesis lives in, and every
hypothesis test in the harness takes it:

| mode | alternative | p-value |
|---|---|---|
| `two-sided` (default) | the value **differs** from the hypothesised one | `2*min(u, 1-u)` |
| `upper` | it is **greater** | `u` |
| `lower` | it is **less** | `1 - u` |

where `u` is the upper-tail probability of the statistic. Getting this
wrong is quiet: a two-sided p-value used for a one-sided question is
exactly double, and an upper tail used for a two-sided one is exactly
half. Both still look like probabilities.

The expected values below are computed here with scipy, so the fleet's
three answers can be checked against them one at a time.


In [7]:
from scipy import stats

print(f"n = {n}   sigma0_sq = {SIGMA0_SQ}   df = {n - 1}")
print(f"{'column':<18} {'chi2':>14} {'upper':>14} {'lower':>14} {'two-sided':>14}")
for c in SHARED:
    chi2 = (n - 1) * pooled[c].var(ddof=1) / SIGMA0_SQ
    u = stats.chi2.sf(chi2, n - 1)
    print(f"{c:<18} {chi2:>14.4f} {u:>14.6g} {1-u:>14.6g} {2*min(u, 1-u):>14.6g}")


n = 380   sigma0_sq = 1.0   df = 379
column                       chi2          upper          lower      two-sided
vibration_rms            135.8601              1              0              0
shaft_speed_rpm      5499076.0906              0              1              0
power_kw               43305.7066              0              1              0


### Ask the fleet, once per tail

In [8]:
for mode in ("two-sided", "upper", "lower"):
    answer = await ask_scarlet_agent(
        f"Run a chi-squared test for a variance across the fleet using the "
        f"chi2_test skill with sigma0_sq={SIGMA0_SQ} and mode='{mode}'. "
        f"Report the chi2 statistic, the degrees of freedom and the "
        f"p-value for each column, and say which mode was used. Do not "
        f"name any data source or column yourself.")
    print(f"===== mode={mode} =====")
    print(answer)
    print()


===== mode=two-sided =====
The chi-squared test for variance was run with `sigma0_sq = 1.0` and `mode = 'two-sided'`.

Results per column:

- **power_kw**:  
  - χ² statistic = 43305.71  
  - degrees of freedom = 379  
  - p-value = 0.0  

- **shaft_speed_rpm**:  
  - χ² statistic = 5499076.09  
  - degrees of freedom = 379  
  - p-value = 0.0  

- **vibration_rms**:  
  - χ² statistic = 135.86  
  - degrees of freedom = 379  
  - p-value = 0.0  

All p-values are effectively 0 (except for `vibration_rms`, where the two-sided p-value is 0.0, and the upper tail is 1.0, indicating the observed variance is far below the hypothesised variance of 1.0). This suggests the fleet's variance for all three columns significantly differs from the hypothesised variance of 1.0.



===== mode=upper =====
The chi-squared test for variance was run in **upper** mode with hypothesised variance σ₀² = 1.0.

Results per column:

- **power_kw**:  
  - χ² statistic = 43305.71  
  - degrees of freedom = 379  
  - p-value = 0.0  

- **shaft_speed_rpm**:  
  - χ² statistic = 5499076.09  
  - degrees of freedom = 379  
  - p-value = 0.0  

- **vibration_rms**:  
  - χ² statistic = 135.86  
  - degrees of freedom = 379  
  - p-value = 1.0  

Interpretation (upper-tail test):  
- For **power_kw** and **shaft_speed_rpm**, the p-values are 0.0, indicating overwhelming evidence that the true variance exceeds 1.0.  
- For **vibration_rms**, the p-value is 1.0, indicating no evidence that the variance exceeds 1.0 (the observed variance is ≤ 1.0).



===== mode=lower =====
The chi-squared test for variance was run in **lower** mode (testing whether the fleet's variance is significantly less than the hypothesised variance of 1.0).

Results per column:

- **power_kw**:  
  - χ² statistic = 43305.71  
  - degrees of freedom = 379  
  - p-value = 1.0  

- **shaft_speed_rpm**:  
  - χ² statistic = 5499076.09  
  - degrees of freedom = 379  
  - p-value = 1.0  

- **vibration_rms**:  
  - χ² statistic = 135.86  
  - degrees of freedom = 379  
  - p-value = 0.0  

Interpretation: In lower-tail mode, a large p-value (close to 1.0) means we fail to reject the null hypothesis that the variance is ≥ 1.0; a p-value near 0 would indicate strong evidence that the variance is *less* than 1.0. Only **vibration_rms** shows evidence (p = 0.0) that its variance is significantly *less* than 1.0. The other two columns show no such evidence.



## Teardown

In [9]:
if GUSTAVO_MANAGED_AGENTS:
    print("Gustavo owns this fleet - leaving it alone.")
else:
    print("agent containers stopped and removed")

agent containers stopped and removed
